In [0]:
dbutils.widgets.text("event_blob_path", "")

event_blob_path = dbutils.widgets.get("event_blob_path")

if not event_blob_path:
    raise ValueError(
        "event_blob_path is empty. Run this notebook through ADF."
    )

print("Event blob path:", event_blob_path)

In [0]:
relative_path = event_blob_path.replace("bronze/", "", 1)

bronze_path = (
    "abfss://bronze@stcommercesrc.dfs.core.windows.net/"
    + relative_path
)

print("Bronze path:")
print(bronze_path)

In [0]:
print("STEP 1 - Reading Bronze order file")

df_bronze = spark.read.parquet(bronze_path)

rows_read = df_bronze.count()

print("Rows read from Bronze:", rows_read)

df_bronze.printSchema()

display(df_bronze.limit(10))

**Clean**

In [0]:
from pyspark.sql.functions import col

print("STEP 10.1 - Cleaning Order Items")

df_clean = (
    df_bronze
    .withColumn(
        "unit_price",
        col("unit_price").cast("decimal(12,2)")
    )
    .withColumn(
        "discount_amount",
        col("discount_amount").cast("decimal(12,2)")
    )
    .withColumn(
        "tax_amount",
        col("tax_amount").cast("decimal(12,2)")
    )
    .withColumn(
        "line_total",
        col("line_total").cast("decimal(12,2)")
    )
)

print("Rows before cleaning:", df_bronze.count())
print("Rows after cleaning:", df_clean.count())

display(df_clean.limit(10))

**Duplicates**

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

print("STEP 10.2 - Deduplicating Order Items")

window_spec = (
    Window
    .partitionBy("order_item_id")
    .orderBy(
        col("updated_at").desc_nulls_last(),
        col("created_at").desc_nulls_last()
    )
)

df_ranked = df_clean.withColumn(
    "row_num",
    row_number().over(window_spec)
)

df_deduplicated = (
    df_ranked
    .filter(col("row_num") == 1)
    .drop("row_num")
)

print("Rows before deduplication:", df_clean.count())
print("Rows after deduplication:", df_deduplicated.count())
print(
    "Duplicate rows removed:",
    df_clean.count() - df_deduplicated.count()
)

**Validate**

In [0]:
print("STEP 10.3 - Validating Order Items")

df_valid = df_deduplicated.filter(
    col("order_item_id").isNotNull()
    & col("order_id").isNotNull()
    & col("product_id").isNotNull()
    & col("quantity").isNotNull()
    & (col("quantity") > 0)
    & col("unit_price").isNotNull()
    & (col("unit_price") > 0)
    & col("discount_amount").isNotNull()
    & (col("discount_amount") >= 0)
    & col("tax_amount").isNotNull()
    & (col("tax_amount") >= 0)
    & col("line_total").isNotNull()
    & (col("line_total") >= 0)
)

print("Deduplicated rows:", df_deduplicated.count())
print("Valid Order Items:", df_valid.count())
print("Rejected Order Items:", df_deduplicated.count() - df_valid.count())

display(df_valid.limit(10))

**Checking rejected row**

In [0]:
print("STEP 11.1 - Inspecting rejected Order Items")

display(df_deduplicated.join(
    df_valid.select("order_item_id"),
    on="order_item_id",
    how="left_anti"
))

**Save rejected rows into silver exceptions**

In [0]:
from delta.tables import DeltaTable

print("STEP 11.2 - Saving rejected Order Items")

exception_path = (
    "abfss://silver@stcommercesrc.dfs.core.windows.net/"
    "exceptions/order_items"
)

df_rejected = df_deduplicated.join(
    df_valid.select("order_item_id"),
    on="order_item_id",
    how="left_anti"
)

if df_rejected.limit(1).count() > 0:
    (
        df_rejected.write
        .format("delta")
        .mode("append")
        .save(exception_path)
    )

print("Rejected records saved:", df_rejected.count())

**Merge valid rows into silver**

In [0]:
target_table = "ecommerce.silver.order_items_external"

target = DeltaTable.forName(spark, target_table)

(
    target.alias("t")
    .merge(
        df_valid.alias("s"),
        "t.order_item_id = s.order_item_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Order Items Silver MERGE completed.")

In [0]:
df1 = spark.read.format("delta")\
                    .option("header" , True)\
                    .option("inferSchema" , True)\
                    .load("abfss://silver@stcommercesrc.dfs.core.windows.net/exceptions/order_items/")

display(df1)


order_item_id,order_id,product_id,quantity,unit_price,discount_amount,tax_amount,line_total,created_at,updated_at
1003,24702,1592,-3,32535.40,1180.11,5643.95,36999.24,2024-12-15T01:26:51.000Z,2025-01-09T01:26:51.000Z
1002,41554,1167,-3,65197.99,4798.38,22607.57,148205.17,2024-05-24T19:05:27.000Z,2024-06-12T19:05:27.000Z
1001,19573,2817,-3,50868.49,8396.44,16801.30,110141.84,2025-09-25T18:59:43.000Z,2025-10-15T18:59:43.000Z
